In [0]:
from pyspark.sql import functions as F

# GOLD principle: consumption-ready. Drop lineage/meta columns the model
# shouldn't see; keep exactly the label + features. One table = one purpose.
gold = (spark.table("workspace.default.silver_credit")
        .drop("row_id", "_ingested_at", "_source_file", "_silver_processed_at"))
gold.write.mode("overwrite").saveAsTable("workspace.default.gold_credit_features")
print("gold_credit_features:", gold.count(), "rows,", len(gold.columns), "cols")

gold_credit_features: 149999 rows, 12 cols


In [0]:
import mlflow, mlflow.sklearn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# 150K rows fits comfortably in memory -> toPandas() and reuse the exact
# modelling recipe from the calculator project (D5, D6 unchanged).
pdf = spark.table("workspace.default.gold_credit_features").toPandas()
X = pdf.drop(columns="SeriousDlqin2yrs"); y = pdf["SeriousDlqin2yrs"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

mlflow.set_experiment("/Users/sanjana.somashekar1999@gmail.com/credit_scorecard")

with mlflow.start_run(run_name="logreg_baseline"):
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    model.fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

    # This is the MLflow value proposition in three lines: params, metrics,
    # and the model artifact itself - versioned, comparable, reproducible.
    mlflow.log_param("model_type", "logistic_regression")
    mlflow.log_param("test_size", 0.3)
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("gini", 2*auc - 1)
    mlflow.sklearn.log_model(model, "model")
    print(f"AUC {auc:.4f}  Gini {2*auc-1:.4f}")

2026/07/22 13:15:51 INFO mlflow.tracking.fluent: Experiment with name '/Users/sanjana.somashekar1999@gmail.com/credit_scorecard' does not exist. Creating a new experiment.
2026/07/22 13:15:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-779e4a14-4286.cloud.databricks.com/ml/experiments/163414314531444/models/m-3fe22025a86e4003ba9dc1b6debf691e?o=7474659035631480
2026/07/22 13:15:58 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.


AUC 0.8193  Gini 0.6386


In [0]:
with mlflow.start_run(run_name="logreg_balanced_challenger"):
    m2 = make_pipeline(StandardScaler(),
                       LogisticRegression(max_iter=1000, class_weight="balanced"))
    m2.fit(X_tr, y_tr)
    auc2 = roc_auc_score(y_te, m2.predict_proba(X_te)[:, 1])
    mean_pd = m2.predict_proba(X_te)[:, 1].mean()
    mlflow.log_param("model_type", "logreg_class_weight_balanced")
    mlflow.log_metric("auc", auc2)
    mlflow.log_metric("mean_predicted_pd", float(mean_pd))
    print(f"challenger AUC {auc2:.4f}  mean predicted PD {mean_pd:.4f}")

challenger AUC 0.8544  mean predicted PD 0.3515
